#### environment: !pip -q install torch torchvision einops

## GET, Summer 26 Project-Diffusion

You will be implementing a DDPM on MNIST dataset, with and without classifier free guidance. Before starting, make sure you have a good understanding of class notes. All notations are from this tutorial: https://arxiv.org/abs/2208.11970 . We follow the epsilon-estimation formulation.

#### What's provided:
We have provided a diffusion denoising U-Net, a diffusion scheduler and helper funcionts, visualization code for conditional and unconditional generation.

#### Items you need to complete:
Some functions in this starter code are incomplete. You will complete those functions and upload the completed code (in `.ipynb` format) to Gradescope. We will test your implementation with custom inputs.  
   You are required to complete the following functions:
   1. DDPM epsilon estimation loss (`ddpm_loss_epsilon()`).
   2. DDPM epsilon estimation loss for classifier free guidance (`ddpm_loss_cfg()`).  
   3. DDPM unconditional sampling`sample_unconditional()`
   4. DDPM sampling with classifer free guidance`sample_conditional()`

We also provide the visualization code for the sampling. You need to write a report about CFG sampling when you tune the gamma parameter.

In [5]:
import math, random, os, numpy as np  # 1. 수학, 무작위, OS, 데이터 처리를 위한 핵심 모듈 임포트
import torch, torch.nn as nn, torch.nn.functional as F  # 2. PyTorch 딥러닝 프레임워크 및 파이프라인 모듈 임포트
from torch.utils.data import DataLoader  # 3. 데이터 로더 모듈 임포트 (배치 구성 및 데이터 셔플링 담당)
from torchvision import datasets, transforms, utils as vutils  # 4. TorchVision 데이터셋 및 이미지 처리 유틸리티 임포트
from einops import rearrange  # 5. 텐서 재배열 모듈 einops 임포트

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # 6. 연산 하드웨어 장치 설정 (CUDA GPU 사용 가능 여부 자동 감지)
torch.manual_seed(0); np.random.seed(0); random.seed(0)  # 7. 재현성(Reproducibility)을 위한 난수 생성 시드 0 고정

transform = transforms.Compose([  # 8. MNIST 이미지 데이터 전처리 정의 (0~1 픽셀값을 [-1, 1] 범위로 텐서 정규화)
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)  # 9. MNIST 훈련 데이터셋 다운로드 및 로드 (root: ./data)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2, drop_last=True)  # 10. 배치 크기 64, 셔플 활성화, 2개 데이터 로딩 프로세스로 DataLoader 설정
num_classes = 10  # 11. MNIST 클래스 수 설정 (숫자 0~9 총 10개 클래스)

### DDPM with linear scheduler, you only need to run the code, make sure you can understand the code

In [ ]:
class DiffusionSchedule:  # Diffusion schedule 스케줄러 클래스 정의
    def __init__(self, T=200, beta_start=1e-4, beta_end=0.02):  # 1. 초기화 함수 (총 타임스텝 T=200, beta 시작 1e-4, beta 종료 0.02)
        self.T = T  # 2. 총 타임스텝 계수 저장
        betas = torch.linspace(beta_start, beta_end, T)  # 3. 선형 간격으로 Beta 스케줄 텐서 생성
        alphas = 1.0 - betas  # 4. alpha_t = 1 - beta_t 계산
        alphas_bar = torch.cumprod(alphas, dim=0)  # 5. alpha_bar_t = cumprod(alphas) 누적 곱 계산

        self.register(betas, alphas, alphas_bar)  # 6. 디바이스 등록 함수 호출

    def register(self, betas, alphas, alphas_bar):  # 7. 사전 계산 계수 텐서 등록 및 변환 메서드
        self.betas = betas.to(device)  # 8. betas 텐서를 지정 디바이스로 이동
        self.alphas = alphas.to(device)  # 9. alphas 텐서를 지정 디바이스로 이동
        self.alphas_bar = alphas_bar.to(device)  # 10. alphas_bar 텐서를 지정 디바이스로 이동
        self.sqrt_alphas = torch.sqrt(self.alphas)  # 11. sqrt(alpha_t) 계수 계산
        self.sqrt_one_minus_alphas = torch.sqrt(1.0 - self.alphas)  # 12. sqrt(1 - alpha_t) 계수 계산
        self.sqrt_alphas_bar = torch.sqrt(self.alphas_bar)  # 13. sqrt(alpha_bar_t) 계수 계산 (Forward Process의 원본 이미지 비중)
        self.sqrt_one_minus_alphas_bar = torch.sqrt(1.0 - self.alphas_bar)  # 14. sqrt(1 - alpha_bar_t) 계수 계산 (Forward Process의 노이즈 비중)
        self.one_over_sqrt_alphas = 1.0 / self.sqrt_alphas  # 15. 1 / sqrt(alpha_t) 역수 계수 계산

    def sample_timesteps(self, bsz):  # 16. 배치 크기 bsz만큼 임의의 타임스텝 t 샘플링
        return torch.randint(0, self.T, (bsz,), device=device, dtype=torch.long)  # 17. Uniform(0, T-1) 무작위 타임스텝 생성

    def q_sample(self, x0, t, eps):  # 18. Forward Process: x0 이미지와 노이즈 eps로부터 t 타임스텝의 x_t 산출
        s1 = self.sqrt_alphas_bar[t].view(-1,1,1,1)  # 19. sqrt(alpha_bar_t) 텐서의 차원 확장 (B, 1, 1, 1)
        s2 = self.sqrt_one_minus_alphas_bar[t].view(-1,1,1,1)  # 20. sqrt(1 - alpha_bar_t) 텐서의 차원 확장 (B, 1, 1, 1)
        return s1 * x0 + s2 * eps  # 21. x_t = s1 * x0 + s2 * eps 수식 적용 후 리턴

    def posterior_mean_variance(self, xt, eps_pred, t):  # 22. Reverse Process: x_t와 예측 노이즈 eps_pred로부터 posterior 평균 mu와 분산 var 계산
        alpha_t      = self.alphas[t].view(-1,1,1,1)  # 23. t 스텝의 alpha_t 추출 (B, 1, 1, 1)
        alpha_bar_t  = self.alphas_bar[t].view(-1,1,1,1)  # 24. t 스텝의 alpha_bar_t 추출 (B, 1, 1, 1)

        t_prev = torch.clamp(t-1, min=0)  # 25. 이전 타임스텝 t_prev (min 0 클램프) 계산
        alpha_bar_prev = self.alphas_bar[t_prev].view(-1,1,1,1)  # 26. t-1 스텝의 alpha_bar_{t-1} 추출
        alpha_bar_prev = torch.where((t==0).view(-1,1,1,1), torch.ones_like(alpha_bar_prev), alpha_bar_prev)  # 27. t == 0 인 경우 alpha_bar_{-1} = 1.0 적용

        beta_t = self.betas[t].view(-1,1,1,1)  # 28. beta_t 추출
        beta_tilde_t = ((1 - alpha_bar_prev) / (1 - alpha_bar_t)) * beta_t  # 29. Posterior variance beta_tilde_t 계산

        mean = (xt - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * eps_pred) / torch.sqrt(alpha_t)  # 30. Posterior mean mu 계산

        var = beta_tilde_t  # 31. 분산 값 할당
        return mean, var  # 32. 평균(mean)과 분산(var) 튜플 리턴

### Denoising model architecutre, you only need to run the code

In [4]:
import math, torch, torch.nn as nn, torch.nn.functional as F

def sinusoidal_time_embedding(t, dim):  # 1. Sinusoidal Time Embedding 생성 함수
    half = dim // 2  # 2. 임베딩 차원의 절반 구함
    freqs = torch.exp(-math.log(10000) * torch.arange(0, half, device=t.device).float() / half)  # 3. 주파수 파형 스케일 계산
    args = t.float().unsqueeze(1) * freqs.unsqueeze(0)  # 4. t와의 외적 연산 수행
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)  # 5. sin과 cos 값을 결합하여 시간 임베딩 생성 (B, dim)
    if dim % 2 == 1:  # 6. 차원이 홀수인 경우 패딩 추가
        emb = F.pad(emb, (0,1))
    return emb

class AdaGN(nn.Module):  # 7. Adaptive Group Normalization (AdaGN) 모듈
    def __init__(self, num_channels, cond_dim, groups=8):
        super().__init__()
        self.gn = nn.GroupNorm(groups, num_channels, affine=False)  # 8. 아핀 레이어 없이 GroupNorm 생성
        self.fc = nn.Linear(cond_dim, num_channels * 2)  # 9. 조건 벡터로부터 Scale 및 Shift 예측 선형 레이어
        nn.init.zeros_(self.fc.weight)  # 10. 초기 가중치/편향 0으로 설정
        nn.init.zeros_(self.fc.bias)
    def forward(self, x, cond):
        h = self.gn(x)  # 11. 정규화 적용
        s, b = self.fc(cond).chunk(2, dim=1)  # 12. scale(s)과 shift(b) 분할
        s = s.unsqueeze(-1).unsqueeze(-1)  # 13. 공간 차원으로 텐서 확장
        b = b.unsqueeze(-1).unsqueeze(-1)
        return h * (1 + s) + b  # 14. 적응형 변환 h * (1 + s) + b 리턴

class ResBlock(nn.Module):  # 15. ResBlock 잔차 연결 모듈
    def __init__(self, c_in, c_out, cond_dim, dropout=0.0):
        super().__init__()
        self.in_conv = nn.Conv2d(c_in, c_out, 3, padding=1)
        self.ada1 = AdaGN(c_out, cond_dim)
        self.mid_conv = nn.Conv2d(c_out, c_out, 3, padding=1)
        self.ada2 = AdaGN(c_out, cond_dim)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.out_conv = nn.Conv2d(c_out, c_out, 3, padding=1)
        nn.init.zeros_(self.out_conv.weight)
        nn.init.zeros_(self.out_conv.bias)
        self.skip = nn.Conv2d(c_in, c_out, 1) if c_in != c_out else nn.Identity()

    def forward(self, x, cond):
        h = self.in_conv(x)
        h = F.silu(self.ada1(h, cond))
        h = self.mid_conv(h)
        h = F.silu(self.ada2(h, cond))
        h = self.dropout(h)
        h = self.out_conv(h)
        return F.silu(h + self.skip(x))

class Down(nn.Module):  # 16. 다운샘플링 모듈 (Stride 2 Conv2d)
    def __init__(self, c_in, c_out):
        super().__init__()
        self.conv = nn.Conv2d(c_in, c_out, 3, stride=2, padding=1)
    def forward(self, x):
        return self.conv(x)

class Up(nn.Module):  # 17. 업샘플링 모듈 (Nearest Upsample + Conv2d)
    def __init__(self, c_in, c_out):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='nearest')
        self.conv = nn.Conv2d(c_in, c_out, 3, padding=1)
    def forward(self, x):
        return self.conv(self.up(x))

class SmallUNet(nn.Module):  # 18. Denoising U-Net 신경망 (SmallUNet)
    """
    Improved Small U-Net without attention.
    28x28 -> 14x14 -> 7x7 -> 14x14 -> 28x28
    """
    def __init__(self, num_classes=10, null_class=True, ch=64, t_dim=128, y_dim=128, dropout=0.0):
        super().__init__()
        self.null_id = num_classes if null_class else None  # 19. Null 클래스 토큰 지정 (null_id = 10)
        n_embed = num_classes + (1 if null_class else 0)
        self.class_embed = nn.Embedding(n_embed, y_dim)  # 20. 클래스 임베딩 테이블 생성
        self.t_dim, self.y_dim = t_dim, y_dim
        cond_dim = t_dim + y_dim

        self.inp = nn.Conv2d(1, ch, 3, padding=1)

        self.rb1 = ResBlock(ch, ch, cond_dim, dropout)  # 21. 인코더 경로
        self.down1 = Down(ch, ch*2)
        self.rb2 = ResBlock(ch*2, ch*2, cond_dim, dropout)
        self.down2 = Down(ch*2, ch*4)

        self.rb_mid1 = ResBlock(ch*4, ch*4, cond_dim, dropout)  # 22. 바틀넥 경로
        self.rb_mid2 = ResBlock(ch*4, ch*4, cond_dim, dropout)

        self.up1 = Up(ch*4, ch*2)  # 23. 디코더 경로
        self.rb_up1 = ResBlock(ch*2 + ch*2, ch*2, cond_dim, dropout)
        self.up2 = Up(ch*2, ch)
        self.rb_up2 = ResBlock(ch + ch, ch, cond_dim, dropout)

        self.out = nn.Conv2d(ch, 1, 3, padding=1)  # 24. 출력 Conv

        self.proj_t = nn.Sequential(nn.Linear(t_dim, t_dim*4), nn.SiLU(), nn.Linear(t_dim*4, t_dim))  # 25. 시간 및 클래스 임베딩 프로젝션 MLP
        self.proj_y = nn.Sequential(nn.Linear(y_dim, y_dim*4), nn.SiLU(), nn.Linear(y_dim*4, y_dim))

    def forward(self, x, t, y):
        t_emb = self.proj_t(sinusoidal_time_embedding(t, self.t_dim))  # 26. 시간 임베딩 생성 및 프로젝션
        if y is not None:
            y_emb = self.class_embed(y)
        else:
            y_emb = torch.zeros(x.size(0), self.y_dim, device=x.device)
        y_emb = self.proj_y(y_emb)
        cond = torch.cat([t_emb, y_emb], dim=1)  # 27. 통합 조건 벡터 cond 생성

        h0 = self.inp(x)  # 28. 인코더 처리
        h1 = self.rb1(h0, cond)
        h2 = self.down1(h1)
        h3 = self.rb2(h2, cond)
        h4 = self.down2(h3)
        h5 = self.rb_mid1(h4, cond)
        h6 = self.rb_mid2(h5, cond)

        u1 = self.up1(h6)  # 29. 디코더 및 스킵 연결 처리
        u1 = torch.cat([u1, h3], dim=1)
        u1 = self.rb_up1(u1, cond)
        u2 = self.up2(u1)
        u2 = torch.cat([u2, h1], dim=1)
        u2 = self.rb_up2(u2, cond)

        eps = self.out(u2)  # 30. 최종 노이즈 예측 텐서 리턴
        return eps

### you need to implement the DDPM loss function

In [ ]:
sched = DiffusionSchedule(T=200)  # Losses 구현 모듈
net = SmallUNet(num_classes=num_classes, null_class=True, ch=64).to(device)

def ddpm_loss_epsilon(sched, net, x0, y):
    """
    (A) Standard ε-prediction DDPM loss.
      1) sample t ~ Uniform{0..T-1}
      2) sample eps ~ N(0,I)
      3) build x_t = sqrt(alphabar_t) x0 + sqrt(1-alphabar_t) eps
      4) predict eps_hat = net(x_t, t, y)
      5) MSE between eps_hat and eps
    """
    bsz = x0.size(0)  # 1. 배치 크기 bsz 추출
    t = sched.sample_timesteps(bsz)  # 2. Uniform(0, T-1)에서 무작위 타임스텝 t 샘플링
    eps = torch.randn_like(x0)  # 3. x0와 동일한 차원의 가우시안 노이즈 eps ~ N(0, I) 생성
    xt = sched.q_sample(x0, t, eps)  # 4. q_sample을 사용하여 노이즈가 주입된 이미지 x_t 생성
    eps_hat = net(xt, t, y)  # 5. U-Net 추론으로 노이즈 eps_hat 예측
    loss = F.mse_loss(eps_hat, eps)  # 6. 실제 노이즈(eps)와 예측 노이즈(eps_hat) 간의 MSE 손실 산출
    return loss  # 7. 손실 스칼라 리턴


def ddpm_loss_cfg(sched, net, x0, y, null_id):
    """
    (B) CFG training loss (two forward passes).
      1-3) Same steps as above (ddpm_loss_epsilon)
      4-1) eps_hat_cond = net(x_t, t, y)
      4-2) eps_hat_uncond = net(x_t, t, y_null)
      5) loss = MSE(eps_hat_cond, eps) + MSE(eps_hat_uncond, eps)
    """
    bsz = x0.size(0)  # 1. 배치 크기 bsz 추출
    t = sched.sample_timesteps(bsz)  # 2. 타임스텝 t 샘플링
    eps = torch.randn_like(x0)  # 3. 가우시안 노이즈 eps 생성
    xt = sched.q_sample(x0, t, eps)  # 4. Forward Process x_t 생성
    eps_hat_cond = net(xt, t, y)  # 5. Conditional Forward Pass (실제 레이블 y 사용)
    y_null = torch.full((bsz,), null_id, device=x0.device, dtype=torch.long)  # 6. Unconditional Forward Pass (Null 클래스 레이블 null_id=10 사용)
    eps_hat_uncond = net(xt, t, y_null)
    loss = F.mse_loss(eps_hat_cond, eps) + F.mse_loss(eps_hat_uncond, eps)  # 7. 두 Forward Pass의 MSE 손실 합산
    return loss  # 8. 최종 손실 반환

### training, you just need to run the code

In [ ]:
from tqdm import tqdm
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}
    def update(self, model):
        with torch.no_grad():
            for k, v in model.state_dict().items():
                self.shadow[k].mul_(self.decay).add_(v.detach(), alpha=1-self.decay)
    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)
ema = EMA(net, decay=0.999)

In [ ]:
USE_CFG_LOSS = True  # Train
epochs = 30  # feel free to increase
opt = torch.optim.AdamW(net.parameters(), lr=1e-4)

net.train()
for epoch in (range(1, epochs+1)):
    for x, y in tqdm(train_loader):
        x = x.to(device)
        y = y.to(device)
        if USE_CFG_LOSS:
            loss = ddpm_loss_cfg(sched, net, x, y, null_id=net.null_id)
        else:
            loss = ddpm_loss_epsilon(sched, net, x, y)
# print(loss)

        opt.zero_grad()
        loss.backward()
        opt.step()
        ema.update(net)
    print(f"epoch {epoch}: loss={loss.item():.4f}")

ema.copy_to(net); net.eval()

### Implement the samplers: (you need to implement a few lines of code inside sample_unconditional and sample_conditional)

In [ ]:
@torch.no_grad()  # Samplers 구현 모듈
def sample_unconditional(n=64, steps=None):
    """
    (C) Unconditional ancestral DDPM sampling (no guidance).
      Pseudocode:
        x = N(0,I)
        for t in reversed(range(T)):
            eps_hat = net(x, t, y_null)
            mu, var = sched.posterior_mean_variance(x, eps_hat, t)
            x = mu + sqrt(var)*z if t > 0 else mu
        return x.clamp(-1,1)
    """
    net.eval()  # 1. 모델을 평가(eval) 모드로 설정
    T = sched.T if steps is None else steps  # 2. 샘플링 타임스텝 수 T 설정
    x = torch.randn(n, 1, 28, 28, device=device)  # 3. 표준 정규 분포 가우시안 노이즈 x 생성 (n개)
    y_null = torch.full((n,), net.null_id, device=device, dtype=torch.long)  # 4. 무조건부 토큰 null_id 레이블 텐서 생성

    for ti in reversed(range(T)):  # 5. 역방향 타임스텝 T-1 -> 0 순회
        t = torch.full((n,), ti, device=device, dtype=torch.long)  # 6. 현재 타임스텝 텐서 t 생성
        eps_hat = net(x, t, y_null)  # 7. U-Net 추론으로 무조건부 예측 노이즈 eps_hat 산출
        mu, var = sched.posterior_mean_variance(x, eps_hat, t)  # 8. posterior_mean_variance 함수로 p(x_{t-1} | x_t)의 평균(mu)과 분산(var) 구함
        if ti > 0:  # 9. ti > 0 일 때 가우시안 노이즈 추가 역방향 복원
            x = mu + torch.sqrt(var) * torch.randn_like(x)
        else:
            x = mu
    return x.clamp(-1,1)  # 10. 생성 이미지 범위 [-1, 1] 클램핑 후 반환


@torch.no_grad()
def sample_conditional(label, gamma=3.0, n=64, steps=None):
    """
    (D) Conditional sampling with classifier-free guidance.
    """
    net.eval()  # 1. 모델 평가 모드 전환
    T = sched.T if steps is None else steps  # 2. 총 타임스텝 수 T 설정
    x = torch.randn(n, 1, 28, 28, device=device)  # 3. 초기 가우시안 노이즈 x 생성
    y_lab  = torch.full((n,), int(label), device=device, dtype=torch.long)  # 4. 조건부 클래스 레이블 텐서 생성
    y_null = torch.full((n,), net.null_id, device=device, dtype=torch.long)  # 5. 무조건부 null_id 레이블 텐서 생성

    for ti in reversed(range(T)):  # 6. 역방향 타임스텝 T-1 -> 0 순회
        t = torch.full((n,), ti, device=device, dtype=torch.long)  # 7. 타임스텝 텐서 t 생성
        eps_c = net(x, t, y_lab)  # 8. Conditional Pass (eps_c) 추론
        eps_u = net(x, t, y_null)  # 9. Unconditional Pass (eps_u) 추론
        eps_hat = (1.0 + gamma) * eps_c - gamma * eps_u  # 10. CFG 가이던스 공식 적용: eps_hat = (1 + gamma)*eps_c - gamma*eps_u
        mu, var = sched.posterior_mean_variance(x, eps_hat, t)  # 11. posterior mean & variance 수식 산출
        if ti > 0:  # 12. ti > 0 일 때 노이즈 추가 복원
            x = mu + torch.sqrt(var) * torch.randn_like(x)
        else:
            x = mu
    return x.clamp(-1,1)  # 13. 이미지 범위를 [-1, 1]로 클램핑하여 리턴

In [ ]:
def show_grid(x, nrow=8, title=''):  # Helpers to visualize
    x = (x.clamp(-1,1) + 1) * 0.5  # to [0,1]
    grid = vutils.make_grid(x, nrow=nrow)
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8,8))
    plt.axis('off')
    if title: plt.title(title)
    plt.imshow(grid.permute(1,2,0).cpu())
    plt.show()

### Unconditional Generation

In [ ]:
x_un = sample_unconditional(n=64)  # Unconditional samples
show_grid(x_un, title='Unconditional DDPM')

### Conditional Generation with CFG
Tune this gamma and analyze what different gammas represent. Write a report about this with gamma = [-1, 0, 1, 2, 3, 4, 5]. Show generated images with different gamma and explain the images.

In [ ]:
x_c5 = sample_conditional(label=2, gamma=-1, n=64)  # Conditional samples for a digit
show_grid(x_c5, title='Conditional DDPM (label=5, gamma=1.0)')

### Code the exploratory parts below

In [ ]:
## your code
# =========================================================
# 0.2 Exploration 탐구 과제 코드
# =========================================================

# ---------------------------------------------------------
# Exploration 1: Nearby Latent Codes 샘플링 실험
# ---------------------------------------------------------
@torch.no_grad()
def explore_nearby_latents(label=7, t_step=50, noise_std=0.1, gamma=3.0):
    net.eval()
    
    x0_gen = sample_conditional(label=label, gamma=gamma, n=1)  # 1. 숫자 7 이미지 생성 (x0_gen)
    
    t_tensor = torch.full((1,), t_step, device=device, dtype=torch.long)  # 2. 지정한 타임스텝 t_step까지 노이즈 부여 (Forward q_sample)
    eps_orig = torch.randn_like(x0_gen)
    xt_orig = sched.q_sample(x0_gen, t_tensor, eps_orig)
    
    xt_perturbed = xt_orig + noise_std * torch.randn_like(xt_orig)  # 3. 노이즈 이미지에 섭동(perturbation) 추가
    
    x = xt_perturbed.clone()  # 4. 섭동된 노이즈 이미지를 t_step부터 t=0까지 디노이징 복원
    y_lab  = torch.full((1,), int(label), device=device, dtype=torch.long)
    y_null = torch.full((1,), net.null_id, device=device, dtype=torch.long)
    
    for ti in reversed(range(t_step)):
        t = torch.full((1,), ti, device=device, dtype=torch.long)
        eps_c = net(x, t, y_lab)
        eps_u = net(x, t, y_null)
        eps_hat = (1.0 + gamma) * eps_c - gamma * eps_u
        mu, var = sched.posterior_mean_variance(x, eps_hat, t)
        if ti > 0:
            x = mu + torch.sqrt(var) * torch.randn_like(x)
        else:
            x = mu
            
    x_rec = x.clamp(-1, 1)
    
    res = torch.cat([x0_gen, x_rec], dim=0)  # 결과 비교 출력 (원래 생성 이미지 vs 섭동 후 복원 이미지)
    show_grid(res, nrow=2, title=f'Exploration 1: Original vs Perturbed (t={t_step})')

# ---------------------------------------------------------
# Exploration 2: Diffusion Hole Filling (Inpainting) 실험
# ---------------------------------------------------------
@torch.no_grad()
def explore_hole_filling(label=8, t_step=50, gamma=3.0):
    net.eval()
    
    x0_gen = sample_conditional(label=label, gamma=gamma, n=1)  # 1. 숫자 8 이미지 생성 (x0_gen)
    
    mask = torch.ones_like(x0_gen)  # 2. 이미지 상단 절반 마스킹 (상단 14픽셀 0 처리)
    mask[:, :, :14, :] = 0.0
    x0_masked = x0_gen * mask
    
    t_tensor = torch.full((1,), t_step, device=device, dtype=torch.long)  # 3. 마스킹된 이미지를 t_step까지 노이즈화
    eps = torch.randn_like(x0_gen)
    xt_masked = sched.q_sample(x0_masked, t_tensor, eps)
    
    x = xt_masked.clone()  # 4. t_step부터 t=0까지 디노이징을 통한 Hole Filling 수행
    y_lab  = torch.full((1,), int(label), device=device, dtype=torch.long)
    y_null = torch.full((1,), net.null_id, device=device, dtype=torch.long)
    
    for ti in reversed(range(t_step)):
        t = torch.full((1,), ti, device=device, dtype=torch.long)
        eps_c = net(x, t, y_lab)
        eps_u = net(x, t, y_null)
        eps_hat = (1.0 + gamma) * eps_c - gamma * eps_u
        mu, var = sched.posterior_mean_variance(x, eps_hat, t)
        if ti > 0:
            x = mu + torch.sqrt(var) * torch.randn_like(x)
        else:
            x = mu
            
    x_filled = x.clamp(-1, 1)
    
    res = torch.cat([x0_gen, x0_masked, x_filled], dim=0)  # 결과 비교 출력 (원래 이미지 vs 마스킹 이미지 vs 구멍 채운 이미지)
    show_grid(res, nrow=3, title=f'Exploration 2: Original vs Masked vs Hole-Filled (t={t_step})')

explore_nearby_latents(label=7, t_step=50, noise_std=0.1)  # 실험 실행
explore_hole_filling(label=8, t_step=50)